In [ ]:
%load_ext autoreload
%autoreload 2
import plotly.io as pio

from trader import StockDataLoader

pio.get_chrome()


period = "20y"

tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "JPM",
    "PG",
    "JNJ",
    "SPY",
    "QQQ",
    "XLE",
    "GLD",
    "TLT",
    "SHY",
]

df, train_env_data, val_env_data = StockDataLoader(
    tickers, period=period, macro_tickers=["^VIX", "^TNX", "GLD", "XLK", "XLP"]
).load()

In [ ]:
%reload_ext autoreload


import numpy as np
from stable_baselines3 import PPO

from trader import DirichletPolicy, LivePlotCallback, TraderEnvironment

total_timesteps = 10000

env_config = {
    "budget": np.float32(10000.0),
    "episode_length": 252,
    "history_length": 30,
    "fee_rate": np.float32(0.0005),
    "lmb": np.float32(10),
}


train_env = TraderEnvironment(env_data=train_env_data, **env_config)

val_env = TraderEnvironment(env_data=val_env_data, **env_config)
callback = LivePlotCallback(
    total_timesteps=total_timesteps,
    train_env=train_env,
    val_env=val_env,
    tickers=tickers,
)

model = PPO(
    DirichletPolicy,
    train_env,
    verbose=0,
    device="cuda",
)

trained_model = model.learn(
    total_timesteps=total_timesteps, log_interval=4, callback=callback
)